# MO: Mosaic

**Atlas source reading copy · browser-local**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-Spatial-Analysis/blob/88007a32ab0ac970469af4b0c118209325a1989f/notebooks/58_MO_mosaic.ipynb)



Local calculations and bundled inputs; first browser runtime/package download still needs connectivity. Static inspection is not execution certification.

**Provenance:** `notebooks/58_MO_mosaic.ipynb` at `88007a32ab0ac970469af4b0c118209325a1989f`; original SHA-256 `8160315e4b81a7fe991f1d3ff98f9fd7412b5d855549342a906d84ec75834985`. Source-specific license/notices retained under notices; data and media have separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# MO: Mosaic

**Category:** Raster Data Management

**Concept:** Combines raster tiles into a seamless surface.

This notebook is part of a JupyterLite learning lab inspired by the GISGeography *Periodic Table for Spatial Analysis*. It uses synthetic data so it can run entirely in the browser.

## Learning goals

- Recognize what **Mosaic** tries to answer in a geospatial workflow.
- Run a minimal Python demonstration using points, networks, or rasters.
- Modify one parameter and explain how the result changes.


In [ ]:
# Shared setup for browser-friendly geospatial examples.
# These notebooks avoid heavy compiled GIS dependencies so they run well in JupyterLite/Pyodide.
import math
import random
from collections import defaultdict, deque
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

random.seed(7)
np.random.seed(7)

def make_points(n=60, centers=((2, 2), (7, 5)), spread=0.85):
    """Create clustered x/y observations with simple attributes."""
    pts = []
    for i in range(n):
        cx, cy = centers[i % len(centers)]
        x = np.random.normal(cx, spread)
        y = np.random.normal(cy, spread)
        pts.append({'id': i, 'x': x, 'y': y, 'group': chr(65 + (i % len(centers))), 'value': 10 + 2*x + y + np.random.normal(0, 1)})
    return pd.DataFrame(pts)

def grid_surface(nx=80, ny=60):
    """Synthetic elevation-like raster with hills and noise."""
    x = np.linspace(0, 10, nx)
    y = np.linspace(0, 8, ny)
    X, Y = np.meshgrid(x, y)
    Z = 70*np.exp(-((X-3.0)**2 + (Y-5.5)**2)/5.0) + 45*np.exp(-((X-7.5)**2 + (Y-2.5)**2)/3.0)
    Z += 4*np.sin(X*1.7) + 3*np.cos(Y*2.1)
    return X, Y, Z

def plot_points(df, title='Point layer', value_col='value'):
    fig, ax = plt.subplots(figsize=(6, 5))
    s = ax.scatter(df['x'], df['y'], c=df[value_col], s=55)
    ax.set_title(title)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_aspect('equal', adjustable='box')
    fig.colorbar(s, ax=ax, label=value_col)
    plt.show()

def plot_raster(Z, title='Raster surface'):
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(Z, origin='lower', extent=[0, 10, 0, 8])
    ax.set_title(title)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    fig.colorbar(im, ax=ax)
    plt.show()

def distance(a, b):
    return math.hypot(a[0]-b[0], a[1]-b[1])

def nearest_neighbor(df, facilities):
    rows = []
    for _, r in df.iterrows():
        dists = [(name, distance((r.x, r.y), xy)) for name, xy in facilities.items()]
        name, d = min(dists, key=lambda t: t[1])
        rows.append((name, d))
    out = df.copy()
    out['nearest'] = [r[0] for r in rows]
    out['distance'] = [r[1] for r in rows]
    return out

def moving_average_2d(A, radius=1):
    B = np.zeros_like(A, dtype=float)
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            r0, r1 = max(0, i-radius), min(A.shape[0], i+radius+1)
            c0, c1 = max(0, j-radius), min(A.shape[1], j+radius+1)
            B[i, j] = A[r0:r1, c0:c1].mean()
    return B


## Run the mini-lab

The next cell creates a small synthetic geospatial problem. The example is deliberately compact: focus on the concept, then adapt it to your own data or a richer GIS stack later.


In [ ]:
# MO: Mosaic mini-lab
X, Y, Z = grid_surface()
plot_raster(Z, 'Starting surface for MO: Mosaic')

# A small demonstration that fits this element family.
if 'MO' in ['MA', 'MF']:
    result = np.sqrt(np.maximum(Z, 0)) * 10
    plot_raster(result, 'Map algebra / math transform')
elif 'MO' == 'CON':
    result = (Z > np.percentile(Z, 70)).astype(int)
    plot_raster(result, 'Conditional raster: top 30% cells')
elif 'MO' == 'SU':
    slope_y, slope_x = np.gradient(Z)
    flatness = 1 / (1 + np.hypot(slope_x, slope_y))
    elevation_score = (Z - Z.min()) / (Z.max() - Z.min())
    suitability = 0.65 * flatness + 0.35 * elevation_score
    plot_raster(suitability, 'Suitability = flatness + elevation score')
elif 'MO' in ['TA', 'CN']:
    gy, gx = np.gradient(Z)
    slope = np.hypot(gx, gy)
    plot_raster(slope, 'Terrain derivative: slope')
    plt.figure(figsize=(6, 5)); plt.contour(X, Y, Z, levels=12); plt.title('Contours from surface'); plt.gca().set_aspect('equal'); plt.show()
elif 'MO' in ['RS', 'RG']:
    coarse = Z[::4, ::4]
    smooth = moving_average_2d(Z, radius=2)
    plot_raster(coarse, 'Coarse resample')
    plot_raster(smooth, 'Generalized/smoothed raster')
elif 'MO' in ['BA', 'IC', 'CB', 'IS', 'PS', 'AC', 'SG', 'DN', 'ME', 'PH', 'OB', 'RD']:
    red = (Z - Z.min()) / (Z.max() - Z.min())
    nir = np.clip(red + 0.25*np.sin(X) - 0.15*np.cos(Y), 0, 1)
    ndvi = (nir - red) / (nir + red + 1e-9)
    plot_raster(ndvi, 'Remote-sensing style index')
else:
    smooth = moving_average_2d(Z, radius=1)
    plot_raster(smooth, 'Processed raster for MO')

print('Try changing the grid size, thresholds, weights, or smoothing radius above.')


## Reflection prompts

1. What real dataset could you use to practice **Mosaic**?
2. Which parameter in the code has the biggest effect on the result?
3. What would make this analysis misleading if the input data were noisy, biased, or in the wrong coordinate reference system?

## Next steps

- Compare this notebook with related elements in **Raster Data Management**.
- Replace the synthetic coordinates with a CSV of real locations.
- Add a short markdown note explaining when you would use this method in public health, environment, emergency response, planning, or transportation.
